# Lab 3 — Experiments: Make It Better, Break It, Understand It

You have a working digit network from Lab 1. Now you get to be the **researcher**: change one thing at a time, measure what happens, and explain *why*. That habit — **change one thing, measure, explain** — is the whole job tomorrow when the "images" become **your objects**.

Each **Mission** is a Your-turn task. Run the two setup cells first, then work the missions in order (or jump to whichever grabs you). Keep an eye on the scoreboard at the end — bring your best result to the wrap-up.

> **How to run this:** You're on the JupyterHub GPU server — everything is installed. **Shift+Enter** to run each cell. Heads-up: with the whole class training at once on one shared GPU, keep epochs modest so everyone stays fast.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", DEVICE)


## Setup — data + a reusable train/eval harness

We wrap building, training, and testing into small functions so a whole experiment is just a couple of lines. `build_mlp([128, 64])` makes a network; `train_and_eval(model, name=...)` trains it and records the score in `RESULTS` so we can compare everything at the end.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_ds = datasets.MNIST("data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST("data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=1000, shuffle=False)

RESULTS = {}   # name -> best test accuracy, so we can rank experiments later
print(len(train_ds), "train,", len(test_ds), "test")


In [ ]:
def build_mlp(hidden_sizes):
    """Flatten -> [Linear+ReLU]* -> Linear(->10).  e.g. build_mlp([128, 64])."""
    layers = [nn.Flatten()]
    prev = 28 * 28
    for h in hidden_sizes:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        prev = h
    layers += [nn.Linear(prev, 10)]
    return nn.Sequential(*layers).to(DEVICE)


def _run(model, loader, opt=None):
    train = opt is not None
    model.train() if train else model.eval()
    crit = nn.CrossEntropyLoss()
    correct = total = 0
    torch.set_grad_enabled(train)
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss = crit(out, labels)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return correct / total


def train_and_eval(model, epochs=5, lr=1e-3, name="model"):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    te = 0.0
    for ep in range(1, epochs + 1):
        tr = _run(model, train_loader, opt)
        te = _run(model, test_loader)
        print(f"[{name}] epoch {ep:2d}  train {tr:5.1%}  test {te:5.1%}")
    RESULTS[name] = te
    return te


## Mission 1 — Resize the brain

Lab 1's network had hidden layers `[128, 64]`. Does a **bigger** brain do better? A **smaller** one? Try a few and compare. The first line runs; uncomment the `# TODO` lines (or write your own) and run again.

In [ ]:
model = build_mlp([128, 64]); train_and_eval(model, name="mlp 128-64")

# TODO: a bigger brain
# model = build_mlp([256, 128]); train_and_eval(model, name="mlp 256-128")

# TODO: a tiny brain — how small can you go before it hurts?
# model = build_mlp([16]); train_and_eval(model, name="mlp 16")


## Mission 2 — Add depth

More *layers* isn't automatically better. Add a third hidden layer and watch: does accuracy rise, or does **train** pull ahead of **test** (the network starting to memorize)?

In [ ]:
model = build_mlp([128, 64, 32]); train_and_eval(model, name="mlp 128-64-32")
# TODO: go deeper still, e.g. build_mlp([256, 128, 64, 32]). Better, or diminishing returns?


## Mission 3 — Learning rate

The **learning rate** is the step size for each tune-up. Too big overshoots and bounces; too small crawls. Sweep a few and watch the final accuracy — and whether it even trained.

In [ ]:
for lr in [1e-2, 1e-3, 1e-4]:
    m = build_mlp([128, 64])
    train_and_eval(m, epochs=3, lr=lr, name=f"lr={lr}")
    print()


## Mission 4 — Go convolutional (the big jump)

Fully-connected layers ignore *where* pixels are. A **convolutional** network slides small filters across the image and respects its 2-D shape — the same idea behind tomorrow's object recognizer. Train the CNN below; you should sail past **99%**.

**Your turn:** before you run it, read the two conv blocks and predict the tensor size going into the final `Linear`. Each `MaxPool2d(2)` halves the width and height: 28 → 14 → 7. With `c2` channels of 7×7, that's `c2 * 7 * 7` numbers. Then try bumping `c1`/`c2` up — do more filters help?

In [ ]:
class DigitCNN(nn.Module):
    def __init__(self, c1=16, c2=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, c1, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 28 -> 14
            nn.Conv2d(c1, c2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 14 -> 7
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(c2 * 7 * 7, 10))

    def forward(self, x):
        return self.head(self.conv(x))

model = DigitCNN().to(DEVICE)
train_and_eval(model, epochs=3, name="cnn 16-32")
# TODO: try DigitCNN(c1=32, c2=64). More filters -> better? slower?


## Mission 5 — Confusion hunt

Which digit does your network mix up most? Build a 10×10 table: **rows = true digit, columns = what it guessed**. Big numbers *off* the diagonal are the confusable pairs (classic culprits: 4/9, 3/5, 7/1).

This uses whatever `model` you trained last — run it right after a training cell to inspect that model.

In [ ]:
try:
    model
except NameError:
    model = build_mlp([128, 64]); train_and_eval(model, name="mlp 128-64")

cm = torch.zeros(10, 10, dtype=torch.int)
model.eval(); torch.set_grad_enabled(False)
for imgs, labels in test_loader:
    preds = model(imgs.to(DEVICE)).argmax(1).cpu()
    for t, p in zip(labels, preds):
        cm[t, p] += 1

plt.figure(figsize=(5.5, 5.5))
plt.imshow(cm, cmap="Blues")
plt.xlabel("guessed"); plt.ylabel("true")
plt.xticks(range(10)); plt.yticks(range(10))
for i in range(10):
    for j in range(10):
        if cm[i, j] and i != j:
            plt.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8)
plt.title("Confusion matrix (off-diagonal = mistakes)")
plt.tight_layout(); plt.show()

off = cm.clone(); off.fill_diagonal_(0)
i, j = divmod(int(off.argmax()), 10)
print(f"Most confused: true {i} guessed as {j}  ({int(off[i, j])} times)")


## Mission 6 (stretch) — Make it overfit on purpose

**Overfitting** = memorizing the training data instead of learning the pattern. Force it: train on only **200** images for many epochs. Watch **train** accuracy shoot toward 100% while **test** accuracy lags far behind — that gap *is* overfitting. This is the exact failure the lecture's train / validation / test split exists to catch.

In [ ]:
small = Subset(train_ds, range(200))
small_loader = DataLoader(small, batch_size=32, shuffle=True)

model = build_mlp([128, 64])
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print("epoch   train    test    <- watch the gap grow")
for ep in range(1, 31):
    _run(model, small_loader, opt)
    if ep % 5 == 0:
        tr = _run(model, small_loader)
        te = _run(model, test_loader)
        print(f"{ep:5d}   {tr:6.1%}  {te:6.1%}")


## Scoreboard

Everything you tried, ranked by test accuracy. Which change bought the biggest jump? Bring your best line to the wrap-up.

In [ ]:
print("Your experiments (best test accuracy each):\n")
for name, acc in sorted(RESULTS.items(), key=lambda kv: -kv[1]):
    print(f"  {acc:6.1%}   {name}")
